# 02 — Exploratory analysis and stationarity

Daily and weekly profiles, seasonal strength, ADF and KPSS.

**Report sections fed:** 3 (Exploratory analysis and stationarity).


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


## Series overview\n\nFigure 1 of the report.

In [ ]:
fig = plotting.plot_series_overview(y)
fig.savefig("../reports/figures/series_overview.png", dpi=200, bbox_inches="tight")


### Daily profile — trough, peak and their ratio

In [ ]:
profile = y.groupby(y.index.hour).mean()
print(f"trough hour {profile.idxmin()} at {profile.min():.1f}")
print(f"peak   hour {profile.idxmax()} at {profile.max():.1f}")
print(f"peak / trough ratio {profile.max() / profile.min():.2f}")


### Weekday against weekend

In [ ]:
weekend = y[y.index.dayofweek >= 5].mean()
weekday = y[y.index.dayofweek < 5].mean()
print(f"weekday {weekday:.1f}   weekend {weekend:.1f}   ratio {weekend / weekday:.3f}")


## Seasonal strength\n\nSTL-based measure of Wang, Smith and Hyndman (2006).

In [ ]:
for period, label in [(config.DAILY_PERIOD, "daily"), (config.WEEKLY_PERIOD, "weekly")]:
    s = stationarity.seasonal_strength(y_train, period)
    print(f"{label:<7} (period {period:>3}): {s:.3f}")


## Stationarity

ADF and KPSS test complementary nulls. Run on the **training sample only** —
including the test period would let the split influence the specification.


In [ ]:
report = stationarity.stationarity_report(y_train, period=config.DAILY_PERIOD)
report.round(4)


### ACF and PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
plot_acf(y_train, lags=200, ax=axes[0])
plot_pacf(y_train, lags=72, ax=axes[1], method="ywm")
fig.tight_layout()
